# FIFA World Cup 2026 Match Prediction

**Machine Learning Project — Group Stage Match Prediction**

This notebook uses historical international results, team-strength ratings, recent-form features, and multiple regression models to predict match scores and outcomes for the 2026 FIFA World Cup group stage.

# loading datasets

In [183]:
from pathlib import Path

In [ ]:
print("Current working directory:")
print(Path.cwd())

print("\nFound results.csv files:")
candidates = list(Path("data").rglob("results.csv"))

for i, path in enumerate(candidates):
    print(i, "->", path)

# observing datasets

`results`: main train dataset

`scheduled_2026`: for predicting

`fifa_ranking_2026`: lookup table for team strength

In [185]:
import pandas as pd

results = pd.read_csv("data/international_results-master/international_results-master/results.csv")
results["date"] = pd.to_datetime(results["date"])

schedule_2026 = pd.read_csv("data/martj_kaggle/schedule_2026.csv")
schedule_2026["Date"] = pd.to_datetime(schedule_2026["Date"])

ranking_2026 = pd.read_csv("data/martj_kaggle/fifa_ranking_2026-06-08.csv")

elo = pd.read_csv("data/International Football Elo Ratings (1872-2025)/eloratings.csv")

print("results:", results.shape)
print("schedule_2026:", schedule_2026.shape)
print("ranking_2026:", ranking_2026.shape)
print("elo ranking:", elo.shape)

results: (49477, 9)
schedule_2026: (72, 10)
ranking_2026: (211, 8)
elo ranking: (6678, 4)


In [186]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [187]:
results["tournament"].value_counts()

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
                                        ...  
Copa Confraternidad                         1
Benedikt Fontana Cup                        1
ConIFA Challenger Cup                       1
CONIFA World Cup qualification              1
South Asian Super Cup                       1
Name: count, Length: 200, dtype: int64

In [188]:
schedule_2026.head()

,Round,Day,Date,Time,Score,Referee,Notes,Year,home_team,away_team
0,Group stage,Thu,2026-06-11,13:00 (22:00),NaN,NaN,NaN,2026,Mexico,South Africa
1,Group stage,Thu,2026-06-11,20:00 (05:00),NaN,NaN,NaN,2026,Korea Republic,Czechia
2,Group stage,Fri,2026-06-12,15:00 (22:00),NaN,NaN,NaN,2026,Canada,Bosnia-Herzegovina
3,Group stage,Fri,2026-06-12,18:00 (04:00),NaN,NaN,NaN,2026,United States,Paraguay
4,Group stage,Sat,2026-06-13,12:00 (22:00),NaN,NaN,NaN,2026,Qatar,Switzerland


In [189]:
schedule_2026[["Score", "Referee", "Notes"]].value_counts()

Series([], Name: count, dtype: int64)

In [190]:
schedule_2026.drop(columns=["Score", "Referee", "Notes"])

,Round,Day,Date,Time,Year,home_team,away_team
0,Group stage,Thu,2026-06-11,13:00 (22:00),2026,Mexico,South Africa
1,Group stage,Thu,2026-06-11,20:00 (05:00),2026,Korea Republic,Czechia
2,Group stage,Fri,2026-06-12,15:00 (22:00),2026,Canada,Bosnia-Herzegovina
3,Group stage,Fri,2026-06-12,18:00 (04:00),2026,United States,Paraguay
4,Group stage,Sat,2026-06-13,12:00 (22:00),2026,Qatar,Switzerland
...,...,...,...,...,...,...,...
67,Group stage,Sat,2026-06-27,17:00 (00:00),2026,Croatia,Ghana
68,Group stage,Sat,2026-06-27,19:30 (02:30),2026,Colombia,Portugal
69,Group stage,Sat,2026-06-27,19:30 (02:30),2026,Congo DR,Uzbekistan
70,Group stage,Sat,2026-06-27,21:00 (05:00),2026,Jordan,Argentina


In [191]:
ranking_2026.head()

,team,team_code,association,rank,previous_rank,points,previous_points,rated_matches
0,Argentina,ARG,CONMEBOL,1,3,1876.118331,1874.814835,59
1,Spain,ESP,UEFA,2,2,1873.013187,1876.395199,56
2,France,FRA,UEFA,3,1,1869.428449,1877.322731,57
3,England,ENG,UEFA,4,4,1827.048678,1825.965482,57
4,Portugal,POR,UEFA,5,5,1766.177547,1763.834406,56


In [192]:
elo.head()

,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8


## cleaning resutl df

In [193]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


1. creating target labels

In [194]:
df = results.copy()

df["date"] = pd.to_datetime(df["date"])

df["home_goals"] = pd.to_numeric(df["home_score"])
df["away_goals"] = pd.to_numeric(df["away_score"])
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_goals,away_goals
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0.0,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,4.0,2.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,2.0,1.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2.0,2.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,3.0,0.0


2. reault label (for acc check)

In [195]:
import numpy as np


df["result"] = np.select(
    [df["home_goals"] > df["away_goals"], df["home_goals"] < df["away_goals"], df["home_goals"] == df["away_goals"]],
    ["home", "away", "draw"],
    default="unkown")

df["result"].value_counts()

result
home      24242
away      13974
draw      11249
unkown       12
Name: count, dtype: int64

3. dropping nan values in scores

In [196]:
print(df["home_score"].isna().sum())
df[df["result"] == "unkown"]

12


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_goals,away_goals,result
49465,2026-06-26,Egypt,Iran,NaN,NaN,FIFA World Cup,Seattle,United States,True,NaN,NaN,unkown
49466,2026-06-26,New Zealand,Belgium,NaN,NaN,FIFA World Cup,Vancouver,Canada,True,NaN,NaN,unkown
49467,2026-06-26,Cape Verde,Saudi Arabia,NaN,NaN,FIFA World Cup,Houston,United States,True,NaN,NaN,unkown
49468,2026-06-26,Uruguay,Spain,NaN,NaN,FIFA World Cup,Zapopan,Mexico,True,NaN,NaN,unkown
49469,2026-06-26,Norway,France,NaN,NaN,FIFA World Cup,Foxborough,United States,True,NaN,NaN,unkown
49470,2026-06-26,Senegal,Iraq,NaN,NaN,FIFA World Cup,Toronto,Canada,True,NaN,NaN,unkown
49471,2026-06-27,Algeria,Austria,NaN,NaN,FIFA World Cup,Kansas City,United States,True,NaN,NaN,unkown
49472,2026-06-27,Jordan,Argentina,NaN,NaN,FIFA World Cup,Arlington,United States,True,NaN,NaN,unkown
49473,2026-06-27,Colombia,Portugal,NaN,NaN,FIFA World Cup,Miami Gardens,United States,True,NaN,NaN,unkown
49474,2026-06-27,DR Congo,Uzbekistan,NaN,NaN,FIFA World Cup,Atlanta,United States,True,NaN,NaN,unkown


In [197]:
df.shape

(49477, 12)

In [198]:
train_df = df.dropna(subset=["home_score", "away_score"])
train_df.shape

(49465, 12)

In [199]:
train_df = train_df.sort_values(["date", "home_team", "away_team"]).reset_index(drop=True)

4. handling `elo`  

In [200]:
elo["date"] = pd.to_datetime(elo["date"], format="mixed")
elo["rating"] = pd.to_numeric(elo["rating"])
elo["team"] = elo["team"].astype(str).str.strip()

print(elo.columns.tolist())
print(elo["date"].isna().sum(), elo["team"].isna().sum(), elo["rating"].isna().sum(), elo["change"].isna().sum())

['date', 'team', 'rating', 'change']
0 0 31 0


In [201]:
elo = elo.dropna(subset=["rating"]).sort_values(["team", "date"]).reset_index(drop=True)
elo

,date,team,rating,change
0,1977-07-24,Afghanistan,905.0,-11
1,2011-07-03,Afghanistan,837.0,18
2,2014-05-27,Afghanistan,1095.0,-13
3,2015-10-13,Afghanistan,1076.0,-6
4,2016-03-29,Afghanistan,1151.0,24
...,...,...,...,...
6642,2021-10-09,Zimbabwe,1333.0,-7
6643,2025-06-07,Zimbabwe,1360.0,-10
6644,2025-06-10,Zimbabwe,1398.0,38
6645,2025-09-09,Zimbabwe,1355.0,-24


5. merging `elo` with `train_df`

    `home_elo_before` = home team rating before this match date

    `away_elo_before` = away team rating before this match date

    `elo_diff` = home_elo_before - away_elo_before

    * before the match, not same-day rating, because same-day Elo may already include the result. That would be leakage.

In [202]:
def get_prev_elo(match_df, elo_data, side):
    team_col = f"{side}_team"
    elo_col = f"{side}_elo_before"

    left = (match_df[["date", team_col]].rename(columns={team_col: "team"}).reset_index())

    outputs = []

    for team, team_matches in left.groupby("team", sort=False):
        team_matches = team_matches.sort_values("date")

        team_elo = (elo_data[elo_data["team"].eq(team)][["date", "rating"]].sort_values("date"))

        if team_elo.empty:
            team_matches["rating"] = pd.NA
            merged = team_matches
        else:
            merged = pd.merge_asof( team_matches, team_elo, on="date", direction="backward", allow_exact_matches=False)

        outputs.append(merged[["index", "rating"]])

    return (pd.concat(outputs).set_index("index").sort_index()["rating"].rename(elo_col))

In [203]:
train_df["home_elo_before"] = get_prev_elo(train_df, elo, "home")
train_df["away_elo_before"] = get_prev_elo(train_df, elo, "away")

train_df["elo_diff"] = train_df["home_elo_before"] - train_df["away_elo_before"]

train_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_goals,away_goals,result,home_elo_before,away_elo_before,elo_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0.0,0.0,draw,NaN,NaN,NaN
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,4.0,2.0,home,2003.0,1997.0,6.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,2.0,1.0,home,1986.0,2014.0,-28.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2.0,2.0,draw,2006.0,1994.0,12.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,3.0,0.0,home,1997.0,2003.0,-6.0


## handling nan values in elo rating cols in the merged df

In [204]:
train_df[["home_elo_before", "away_elo_before", "elo_diff"]].isna().mean() * 100

home_elo_before    24.961084
away_elo_before    24.910543
elo_diff           40.915799
dtype: float64

In [205]:
missing_home_teams = train_df.loc[train_df["home_elo_before"].isna(), "home_team"].value_counts().head(20)

missing_away_teams = train_df.loc[train_df["away_elo_before"].isna(), "away_team"].value_counts().head(20)

print("Missing home Elo teams:")
print(missing_home_teams)

print("\nMissing away Elo teams:-----------")
print(missing_away_teams)

Missing home Elo teams:
home_team
South Korea             552
United States           502
Saudi Arabia            433
Costa Rica              363
United Arab Emirates    349
Republic of Ireland     346
Trinidad and Tobago     342
Northern Ireland        339
El Salvador             336
Ivory Coast             323
South Africa            285
Hong Kong               268
DR Congo                222
Burkina Faso            215
Myanmar                 179
Czech Republic          178
New Zealand             170
Russia                  152
North Korea             151
North Macedonia         151
Name: count, dtype: int64

Missing away Elo teams:-----------
away_team
South Korea             458
Trinidad and Tobago     411
Northern Ireland        368
Costa Rica              361
Ivory Coast             316
Saudi Arabia            310
DR Congo                304
United States           291
Republic of Ireland     287
El Salvador             278
United Arab Emirates    263
Russia                  25

In [206]:
missing_team_counts = (missing_home_teams.add(missing_away_teams, fill_value=0).sort_values(ascending=False))

print("Number of teams with missing Elo:", len(missing_team_counts))

Number of teams with missing Elo: 22


In [207]:
elo_teams = set(elo["team"].astype(str).str.strip())

for team in missing_team_counts.index:
    print(team, "->", team in elo_teams)

South Korea -> False
United States -> False
Trinidad and Tobago -> False
Saudi Arabia -> False
Costa Rica -> False
Northern Ireland -> False
Ivory Coast -> False
Republic of Ireland -> False
El Salvador -> False
United Arab Emirates -> False
DR Congo -> False
South Africa -> False
Burkina Faso -> False
Hong Kong -> False
New Zealand -> False
Russia -> True
North Korea -> False
Czech Republic -> False
Myanmar -> True
Sierra Leone -> False
German DR -> False
North Macedonia -> False


In [208]:
from difflib import get_close_matches

elo_team_names = sorted(elo["team"].astype(str).str.strip().unique())

for team in missing_team_counts.index:
    suggestions = get_close_matches(team, elo_team_names, n=5, cutoff=0.3)
    print("\n", team)
    print("Suggestions:", suggestions)


 South Korea
Suggestions: ['South\xa0Korea', 'North\xa0Korea', 'South\xa0Africa', 'South\xa0Vietnam', 'South\xa0Yemen']

 United States
Suggestions: ['United\xa0States', 'United\xa0Arab\xa0Emirates', 'United\xa0Arab\xa0Republic', 'Montserrat', 'Federated\xa0States\xa0of\xa0Micronesia']

 Trinidad and Tobago
Suggestions: ['Trinidad\xa0and\xa0Tobago', 'Serbia\xa0and\xa0Montenegro', 'Bosnia\xa0and\xa0Herzegovina', 'Turkmenistan', 'Burkina\xa0Faso']

 Saudi Arabia
Suggestions: ['Saudi\xa0Arabia', 'Serbia', 'Mauritania', 'Surinam', 'Namibia']

 Costa Rica
Suggestions: ['Costa\xa0Rica', 'Croatia', 'Australia', 'Yugoslavia', 'Czechoslovakia']

 Northern Ireland
Suggestions: ['Northern\xa0Ireland', 'Netherlands', 'Northern\xa0Mariana\xa0Islands', 'Ireland', 'North\xa0Yemen']

 Ivory Coast
Suggestions: ['Ivory\xa0Coast', 'Croatia', 'Honduras', 'Iraq', 'Iran']

 Republic of Ireland
Suggestions: ['Ireland', 'Khmer\xa0Republic', 'Democratic\xa0Republic\xa0of\xa0Congo', 'Iceland', 'Northern\xa0Ire

* \xa0 is a non-breaking space. It looks like a normal space, but Python treats it as different.

In [209]:
def clean_team_name(s):
    return str(s).replace("\xa0", " ").strip()

train_df["home_team"] = train_df["home_team"].apply(clean_team_name)
train_df["away_team"] = train_df["away_team"].apply(clean_team_name)
elo["team"] = elo["team"].apply(clean_team_name)

In [210]:
elo_teams = set(elo["team"])

for team in missing_team_counts.index:
    print(team, "->", clean_team_name(team) in elo_teams)

South Korea -> True
United States -> True
Trinidad and Tobago -> True
Saudi Arabia -> True
Costa Rica -> True
Northern Ireland -> True
Ivory Coast -> True
Republic of Ireland -> False
El Salvador -> True
United Arab Emirates -> True
DR Congo -> False
South Africa -> True
Burkina Faso -> True
Hong Kong -> True
New Zealand -> True
Russia -> True
North Korea -> True
Czech Republic -> False
Myanmar -> True
Sierra Leone -> True
German DR -> False
North Macedonia -> True


In [211]:
train_df["home_elo_before"] = get_prev_elo(train_df, elo, "home")
train_df["away_elo_before"] = get_prev_elo(train_df, elo, "away")

train_df["elo_diff"] = train_df["home_elo_before"] - train_df["away_elo_before"]

In [212]:
train_df[["home_elo_before", "away_elo_before", "elo_diff"]].isna().mean() * 100

home_elo_before    12.143940
away_elo_before    12.940463
elo_diff           20.125341
dtype: float64

In [213]:
missing_home_teams = train_df.loc[train_df["home_elo_before"].isna(), "home_team"].value_counts().head(20)

missing_away_teams = train_df.loc[train_df["away_elo_before"].isna(), "away_team"].value_counts().head(20)

print("Missing home Elo teams:")
print(missing_home_teams)

print("\nMissing away Elo teams:-----------")
print(missing_away_teams)

Missing home Elo teams:
home_team
Republic of Ireland    346
DR Congo               222
Myanmar                179
Czech Republic         178
Russia                 152
Guernsey               132
German DR              132
Moldova                130
Jersey                 129
North Macedonia        118
Northern Ireland       116
Eswatini               105
Austria                 91
Portugal                78
Suriname                75
Macau                   69
Réunion                 67
Maldives                67
Malaysia                61
Vietnam Republic        60
Name: count, dtype: int64

Missing away Elo teams:-----------
away_team
DR Congo               304
Republic of Ireland    287
Russia                 252
Czech Republic         185
German DR              166
Moldova                165
Vietnam Republic       135
Northern Ireland       120
North Macedonia        114
Eswatini               110
Guernsey               108
Jersey                 106
Serbia                  95
Aus

In [214]:
missing_team_counts = (missing_home_teams.add(missing_away_teams, fill_value=0).sort_values(ascending=False))

In [215]:
from difflib import get_close_matches

elo_team_names = sorted(elo["team"].astype(str).str.strip().unique())

for team in missing_team_counts.index:
    suggestions = get_close_matches(team, elo_team_names, n=5, cutoff=0.3)
    print("\n", team)
    print("Suggestions:", suggestions)


 Republic of Ireland
Suggestions: ['Democratic Republic of Congo', 'Ireland', 'Northern Ireland', 'Khmer Republic', 'New Zealand']

 DR Congo
Suggestions: ['Congo', 'Togo', 'Mongolia', 'Hong Kong', 'Tonga']

 Russia
Suggestions: ['Russia', 'Tunisia', 'Austria', 'Australia', 'Yugoslavia']

 Czech Republic
Suggestions: ['Khmer Republic', 'Central African Republic', 'Dominican Republic', 'United Arab Republic', 'Czechia']

 German DR
Suggestions: ['Germany', 'West Germany', 'East Germany', 'Georgia', 'Bermuda']

 Moldova
Suggestions: ['Moldova', 'Bolivia', 'Slovenia', 'Slovakia', 'Mongolia']

 Myanmar
Suggestions: ['Myanmar', 'Panama', 'Guyana', 'Denmark', 'Zanzibar']

 Guernsey
Suggestions: ['Germany', 'Turkey', 'Guinea', 'West Germany', 'Suriname']

 Northern Ireland
Suggestions: ['Northern Ireland', 'Northern Mariana Islands', 'North Korea', 'Netherlands', 'Northern Cyprus']

 Jersey
Suggestions: ['Turkey', 'Germany', 'Belarus', 'Peru', 'Palestine']

 North Macedonia
Suggestions: ['No

In [216]:
elo[elo["team"].str.contains("Congo", case=False, na=False)]["team"].unique()

<StringArray>
['Congo', 'Congo-Brazzaville', 'Democratic Republic of Congo']
Length: 3, dtype: str

In [217]:
for name in ["Ireland", "Czechia", "East Germany", "Macedonia", "Swaziland", "Reunion"]:
    print(name, "->", name in set(elo["team"]))

Ireland -> True
Czechia -> True
East Germany -> True
Macedonia -> True
Swaziland -> True
Reunion -> True


In [218]:
team_name_map = {
    "Republic of Ireland": "Ireland",
    "DR Congo": "Democratic Republic of Congo",
    "Czech Republic": "Czechia",
    "German DR": "East Germany",
    "North Macedonia": "Macedonia",
    "Eswatini": "Swaziland",
    "Réunion": "Reunion",
    "Vietnam Republic": "South Vietnam",
}

train_df["home_team_elo"] = train_df["home_team"].replace(team_name_map)
train_df["away_team_elo"] = train_df["away_team"].replace(team_name_map)

In [219]:
def get_prev_elo(match_df, elo_data, side):
    team_col = f"{side}_team_elo"
    elo_col = f"{side}_elo_before"

    left = (match_df[["date", team_col]].rename(columns={team_col: "team"}).reset_index())

    outputs = []

    for team, team_matches in left.groupby("team", sort=False):
        team_matches = team_matches.sort_values("date")

        team_elo = (elo_data[elo_data["team"].eq(team)][["date", "rating"]].sort_values("date"))

        if team_elo.empty:
            team_matches["rating"] = pd.NA
            merged = team_matches
        else:
            merged = pd.merge_asof( team_matches, team_elo, on="date", direction="backward", allow_exact_matches=False)

        outputs.append(merged[["index", "rating"]])

    return (pd.concat(outputs).set_index("index").sort_index()["rating"].rename(elo_col))

In [220]:
train_df["home_elo_before"] = get_prev_elo(train_df, elo, "home")
train_df["away_elo_before"] = get_prev_elo(train_df, elo, "away")
train_df["elo_diff"] = train_df["home_elo_before"] - train_df["away_elo_before"]

In [221]:
train_df[["home_elo_before", "away_elo_before", "elo_diff"]].isna().mean() * 100

home_elo_before     9.924189
away_elo_before    10.431618
elo_diff           15.706055
dtype: float64

7. filling the remaining nan values in elo using Time-aware team-wise forward fill + neutral value for first missing values + missing indicator.

In [222]:
elo.describe()

,date,rating,change
count,6647,6647.000000,6647.000000
mean,2005-05-28 07:32:59.539642,1490.960734,0.010230
min,1872-11-30 00:00:00,0.000000,-86.000000
25%,2000-10-08 00:00:00,1275.000000,-9.000000
50%,2011-09-06 00:00:00,1511.000000,0.000000
75%,2019-11-19 00:00:00,1712.000000,9.000000
max,2025-12-13 00:00:00,2171.000000,86.000000
std,NaN,299.252797,15.330393


In [223]:
import re

INITIAL_ELO = 2000

# 1) missing flags before filling
train_df["home_elo_missing"] = train_df["home_elo_before"].isna().astype(int)
train_df["away_elo_missing"] = train_df["away_elo_before"].isna().astype(int)

# 2) keep original row order
train_df = train_df.sort_values("date").reset_index(drop=True)
train_df["_row_id"] = train_df.index

# 3) make long team-level table
home_long = train_df[
    ["_row_id", "date", "home_team_elo", "home_elo_before"]].rename(columns={"home_team_elo": "team", "home_elo_before": "elo"})

home_long["side"] = "home"

away_long = train_df[
    ["_row_id", "date", "away_team_elo", "away_elo_before"]].rename(columns={"away_team_elo": "team", "away_elo_before": "elo"})

away_long["side"] = "away"

elo_long = pd.concat([home_long, away_long], ignore_index=True)

# 4) time-aware forward fill per team
elo_long = elo_long.sort_values(["team", "date", "_row_id"])

elo_long["elo_filled"] = elo_long.groupby("team")["elo"].ffill().fillna(INITIAL_ELO)


# 5) put values back into train_df
home_filled = elo_long[elo_long["side"].eq("home")].set_index("_row_id")["elo_filled"]


away_filled = elo_long[elo_long["side"].eq("away")].set_index("_row_id")["elo_filled"]


train_df["home_elo_before"] = train_df["_row_id"].map(home_filled)
train_df["away_elo_before"] = train_df["_row_id"].map(away_filled)

# 6) recompute difference
train_df["elo_diff"] = train_df["home_elo_before"] - train_df["away_elo_before"]

# 7) clean helper column
train_df = train_df.drop(columns="_row_id")

In [224]:
train_df[[ "home_elo_before", "away_elo_before", "elo_diff", "home_elo_missing", "away_elo_missing",]].isna().sum()

home_elo_before     0
away_elo_before     0
elo_diff            0
home_elo_missing    0
away_elo_missing    0
dtype: int64

In [225]:
train_df.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_goals,away_goals,result,home_elo_before,away_elo_before,elo_diff,home_team_elo,away_team_elo,home_elo_missing,away_elo_missing
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0.0,0.0,draw,2000,2000,0,Scotland,England,1,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,4.0,2.0,home,2003.0,1997.0,6.0,England,Scotland,0,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,2.0,1.0,home,1986.0,2014.0,-28.0,Scotland,England,0,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2.0,2.0,draw,2006.0,1994.0,12.0,England,Scotland,0,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,3.0,0.0,home,1997.0,2003.0,-6.0,Scotland,England,0,0
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False,4.0,0.0,home,2010.0,2000,10.0,Scotland,Wales,0,1
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False,1.0,3.0,away,1990.0,2011.0,-21.0,England,Scotland,0,0
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False,0.0,2.0,away,1499.0,2011.0,-512.0,Wales,Scotland,0,0
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False,7.0,2.0,home,2011.0,1990.0,21.0,Scotland,England,0,0
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False,9.0,0.0,home,2011.0,1499.0,512.0,Scotland,Wales,0,0


In [226]:
nan_perc = train_df.isna().mean().mul(100).sort_values(ascending=False)
nan_perc

date                0.0
home_team           0.0
away_team           0.0
home_score          0.0
away_score          0.0
tournament          0.0
city                0.0
country             0.0
neutral             0.0
home_goals          0.0
away_goals          0.0
result              0.0
home_elo_before     0.0
away_elo_before     0.0
elo_diff            0.0
home_team_elo       0.0
away_team_elo       0.0
home_elo_missing    0.0
away_elo_missing    0.0
dtype: float64

In [227]:
train_df.duplicated().sum()

np.int64(0)

In [228]:
model_df = train_df.drop(columns=["home_team_elo", "away_team_elo"])

In [229]:
elo.columns.tolist()

['date', 'team', 'rating', 'change']

## feature engineering

In [230]:
model_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_goals,away_goals,result,home_elo_before,away_elo_before,elo_diff,home_elo_missing,away_elo_missing
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0.0,0.0,draw,2000,2000,0,1,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,4.0,2.0,home,2003.0,1997.0,6.0,0,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,2.0,1.0,home,1986.0,2014.0,-28.0,0,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2.0,2.0,draw,2006.0,1994.0,12.0,0,0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,3.0,0.0,home,1997.0,2003.0,-6.0,0,0


8. home_avg_goals_for_last_5 & away_avg_goals_for_last_5 & home_avg_goals_against_last_5 & away_avg_goals_against_last_5

In [231]:
model_df = model_df.sort_values(["date", "home_team", "away_team"]).reset_index(drop=True)
model_df["_match_id"] = model_df.index

goal_form_cols = ["home_avg_goals_for_last_5", "away_avg_goals_for_last_5", "home_avg_goals_against_last_5", "away_avg_goals_against_last_5",]

model_df = model_df.drop(columns=goal_form_cols, errors="ignore")

home_history = (
    model_df[["_match_id", "date", "home_team", "home_goals", "away_goals"]]
    .rename(columns={"home_team": "team", "home_goals": "goals_for", "away_goals": "goals_against",}))

home_history["side"] = "home"

away_history = (
    model_df[["_match_id", "date", "away_team", "away_goals", "home_goals"]]
    .rename(columns={ "away_team": "team", "away_goals": "goals_for", "home_goals": "goals_against",}))

away_history["side"] = "away"

team_history = pd.concat([home_history, away_history], ignore_index=True)
team_history = team_history.sort_values(["team", "date", "_match_id"])

team_history["avg_goals_for_last_5"] = team_history.groupby("team")["goals_for"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())


team_history["avg_goals_against_last_5"] = team_history.groupby("team")["goals_against"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())


home_features = (
    team_history[team_history["side"].eq("home")].set_index("_match_id")[["avg_goals_for_last_5", "avg_goals_against_last_5"]]
    .rename(columns={"avg_goals_for_last_5": "home_avg_goals_for_last_5",  "avg_goals_against_last_5": "home_avg_goals_against_last_5",}))

away_features = (
    team_history[team_history["side"].eq("away")].set_index("_match_id")[["avg_goals_for_last_5", "avg_goals_against_last_5"]]
    .rename(columns={"avg_goals_for_last_5": "away_avg_goals_for_last_5", "avg_goals_against_last_5": "away_avg_goals_against_last_5",}))

model_df = model_df.join(home_features, on="_match_id")
model_df = model_df.join(away_features, on="_match_id")

INITIAL_GOALS_AVG = model_df[["home_goals", "away_goals"]].stack().mean()

model_df[goal_form_cols] = model_df[goal_form_cols].fillna(INITIAL_GOALS_AVG)

model_df[goal_form_cols].isna().sum()

home_avg_goals_for_last_5        0
away_avg_goals_for_last_5        0
home_avg_goals_against_last_5    0
away_avg_goals_against_last_5    0
dtype: int64

9. home_avg_points_last_5 & away_avg_points_last_5

In [232]:
# make sure match id exists
if "_match_id" not in model_df.columns:
    model_df = model_df.sort_values(["date", "home_team", "away_team"]).reset_index(drop=True)
    model_df["_match_id"] = model_df.index

point_form_cols = ["home_avg_points_last_5", "away_avg_points_last_5",]

model_df = model_df.drop(columns=point_form_cols, errors="ignore")

# points gained in the current match
model_df["home_points"] = np.select(
    [model_df["home_goals"] > model_df["away_goals"], model_df["home_goals"] == model_df["away_goals"], model_df["home_goals"] < model_df["away_goals"],], [3, 1, 0])

model_df["away_points"] = np.select(
    [model_df["away_goals"] > model_df["home_goals"], model_df["away_goals"] == model_df["home_goals"], model_df["away_goals"] < model_df["home_goals"],], [3, 1, 0])

home_points_history = (
    model_df[["_match_id", "date", "home_team", "home_points"]].rename(columns={"home_team": "team", "home_points": "points",}))

home_points_history["side"] = "home"

away_points_history = (
    model_df[["_match_id", "date", "away_team", "away_points"]].rename(columns={"away_team": "team", "away_points": "points",}))

away_points_history["side"] = "away"

points_history = pd.concat([home_points_history, away_points_history], ignore_index=True)
points_history = points_history.sort_values(["team", "date", "_match_id"])

points_history["avg_points_last_5"] = (points_history.groupby("team")["points"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean()))

home_points_feature = (points_history[points_history["side"].eq("home")].set_index("_match_id")["avg_points_last_5"].rename("home_avg_points_last_5"))

away_points_feature = (points_history[points_history["side"].eq("away")].set_index("_match_id")["avg_points_last_5"].rename("away_avg_points_last_5"))

model_df = model_df.join(home_points_feature, on="_match_id")
model_df = model_df.join(away_points_feature, on="_match_id")

INITIAL_POINTS_AVG = model_df[["home_points", "away_points"]].stack().mean()

model_df[point_form_cols] = model_df[point_form_cols].fillna(INITIAL_POINTS_AVG)

model_df[point_form_cols].isna().sum()

home_avg_points_last_5    0
away_avg_points_last_5    0
dtype: int64

10. home_days_since_last_match & away_days_since_last_match

In [233]:
# make sure match id exists
if "_match_id" not in model_df.columns:
    model_df = model_df.sort_values(["date", "home_team", "away_team"]).reset_index(drop=True)
    model_df["_match_id"] = model_df.index

model_df["date"] = pd.to_datetime(model_df["date"])

rest_cols = ["home_days_since_last_match", "away_days_since_last_match",]

model_df = model_df.drop(columns=rest_cols, errors="ignore")

home_rest = model_df[["_match_id", "date", "home_team"]].rename(columns={"home_team": "team"})

home_rest["side"] = "home"

away_rest = model_df[["_match_id", "date", "away_team"]].rename(columns={"away_team": "team"})


away_rest["side"] = "away"

rest_history = pd.concat([home_rest, away_rest], ignore_index=True)
rest_history = rest_history.sort_values(["team", "date", "_match_id"])

rest_history["previous_match_date"] = rest_history.groupby("team")["date"].shift(1)


rest_history["days_since_last_match"] = (rest_history["date"] - rest_history["previous_match_date"]).dt.days

home_rest_feature = (rest_history[rest_history["side"].eq("home")].set_index("_match_id")["days_since_last_match"].rename("home_days_since_last_match"))

away_rest_feature = (rest_history[rest_history["side"].eq("away")].set_index("_match_id")["days_since_last_match"].rename("away_days_since_last_match"))

model_df = model_df.join(home_rest_feature, on="_match_id")
model_df = model_df.join(away_rest_feature, on="_match_id")

INITIAL_REST_DAYS = rest_history["days_since_last_match"].median()

model_df[rest_cols] = model_df[rest_cols].fillna(INITIAL_REST_DAYS)

model_df[rest_cols].isna().sum()

home_days_since_last_match    0
away_days_since_last_match    0
dtype: int64

11. is_home_country and year

In [234]:
model_df["is_home_country"] = (model_df["home_team"].eq(model_df["country"])).astype(int)

model_df["year"] = model_df["date"].dt.year

In [235]:
model_df.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_goals,...,away_avg_goals_for_last_5,away_avg_goals_against_last_5,home_points,away_points,home_avg_points_last_5,away_avg_points_last_5,home_days_since_last_match,away_days_since_last_match,is_home_country,year
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,0.0,...,1.469595,1.469595,1,1,1.386293,1.386293,14.0,14.0,1,1872
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,4.0,...,0.000000,0.000000,3,0,1.000000,1.000000,98.0,98.0,1,1873
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,2.0,...,2.000000,1.000000,3,0,0.500000,2.000000,364.0,364.0,1,1874
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,2.0,...,1.333333,1.666667,1,1,1.333333,1.333333,364.0,364.0,1,1875
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,3.0,...,1.750000,1.500000,3,0,1.250000,1.250000,364.0,364.0,1,1876
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False,4.0,...,1.469595,1.469595,3,0,1.600000,1.386293,21.0,14.0,1,1876
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False,1.0,...,2.600000,1.400000,0,3,1.000000,2.000000,364.0,343.0,1,1877
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False,0.0,...,2.800000,0.800000,0,3,0.000000,2.600000,345.0,2.0,1,1877
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False,7.0,...,1.600000,2.400000,3,0,2.600000,0.800000,362.0,364.0,1,1878
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False,9.0,...,0.000000,3.000000,3,0,3.000000,0.000000,21.0,383.0,1,1878


In [236]:
model_df[["away_goals", "home_goals"]].isna().sum().sum()

np.int64(0)

In [237]:
model_df[["away_score", "home_score"]].isna().sum().sum()

np.int64(0)

12. split the dataset

In [238]:
X = model_df.drop(columns=["home_goals", "away_goals"])

y_home = model_df["home_goals"]
y_away = model_df["away_goals"]

y = model_df[["home_goals", "away_goals"]]

In [239]:
print(X.isna().sum().sum())
print(y_home.isna().sum(), y_away.isna().sum())

0
0 0


13. time-based split

In [240]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import PoissonRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score

In [241]:
model_df = model_df.sort_values("date").reset_index(drop=True)

feature_cols = [
    "home_elo_before",
    "away_elo_before",
    "elo_diff",
    "home_elo_missing",
    "away_elo_missing",

    "home_avg_goals_for_last_5",
    "away_avg_goals_for_last_5",
    "home_avg_goals_against_last_5",
    "away_avg_goals_against_last_5",

    "home_avg_points_last_5",
    "away_avg_points_last_5",

    "home_days_since_last_match",
    "away_days_since_last_match",

    "neutral",
    "is_home_country",
    "year",
]

X = model_df[feature_cols]
y = model_df[["home_goals", "away_goals"]]

print("Missing in X:", X.isna().sum().sum())
print("Missing in y:", y.isna().sum().sum())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

Missing in X: 0
Missing in y: 0


## evaluation function

In [242]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score, confusion_matrix, mean_poisson_deviance, mean_absolute_error, mean_squared_error


def evaluation_model(model_name, yTest, yPrediction, yProb=None):
    yPrediction = np.asarray(yPrediction)
    yPrediction = np.clip(yPrediction, 1e-9, None)

    actual_home_goals = yTest["home_goals"].to_numpy()
    actual_away_goals = yTest["away_goals"].to_numpy()

    pred_home_goals = yPrediction[:, 0]
    pred_away_goals = yPrediction[:, 1]

    home_mae = mean_absolute_error(actual_home_goals, pred_home_goals)
    away_mae = mean_absolute_error(actual_away_goals, pred_away_goals)

    home_rmse = np.sqrt(mean_squared_error(actual_home_goals, pred_home_goals))
    away_rmse = np.sqrt(mean_squared_error(actual_away_goals, pred_away_goals))

    home_r2 = r2_score(actual_home_goals, pred_home_goals)
    away_r2 = r2_score(actual_away_goals, pred_away_goals)

    home_poisson_deviance = mean_poisson_deviance(actual_home_goals, pred_home_goals)
    away_poisson_deviance = mean_poisson_deviance(actual_away_goals, pred_away_goals)

    pred_home_score = np.rint(pred_home_goals).clip(0).astype(int)
    pred_away_score = np.rint(pred_away_goals).clip(0).astype(int)

    actual_home_score = actual_home_goals.astype(int)
    actual_away_score = actual_away_goals.astype(int)

    exact_score_accuracy = np.mean((pred_home_score == actual_home_score) & (pred_away_score == actual_away_score))

    actual_result = np.where(actual_home_score > actual_away_score, "home", np.where(actual_home_score < actual_away_score, "away", "draw"))

    pred_result = np.where(pred_home_score > pred_away_score, "home", np.where(pred_home_score < pred_away_score, "away", "draw"))

    acc = accuracy_score(actual_result, pred_result)
    precision = precision_score(actual_result, pred_result, average="weighted", zero_division=0)
    recall = recall_score(actual_result, pred_result, average="weighted", zero_division=0)
    fscore = f1_score(actual_result, pred_result, average="weighted", zero_division=0)

    return (
        f"{model_name}: "
        f"home MAE: {home_mae}, "
        f"away MAE: {away_mae}, "
        f"home RMSE: {home_rmse}, "
        f"away RMSE: {away_rmse}, "
        f"home R2: {home_r2}, "
        f"away R2: {away_r2}, "
        f"home poisson deviance: {home_poisson_deviance}, "
        f"away poisson deviance: {away_poisson_deviance}, "
        f"exact score accuracy: {exact_score_accuracy}, "
        f"result accuracy: {acc}, "
        f"precision: {precision}, "
        f"recall: {recall}, "
        f"Fscore: {fscore}"
    )

## model training

# 1. Poisson regression

In [243]:
poisson_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MultiOutputRegressor(
        PoissonRegressor(alpha=0.1, max_iter=1000)))])

poisson_model.fit(X_train, y_train)

pred_goals = poisson_model.predict(X_test)

print(evaluation_model("Poisson Goal Model", y_test, pred_goals))

Poisson Goal Model: home MAE: 1.0543651266914442, away MAE: 0.8564527279754135, home RMSE: 1.4085193593836758, away RMSE: 1.1750067310998993, home R2: 0.25198173728550455, away R2: 0.2000169331437176, home poisson deviance: 1.264173866427537, away poisson deviance: 1.221694725371126, exact score accuracy: 0.10522591731527342, result accuracy: 0.5219852420903669, precision: 0.6065047453707555, recall: 0.5219852420903669, Fscore: 0.5237656417286375


# 2. dummy

In [244]:
from sklearn.dummy import DummyRegressor

dummy_model = Pipeline([
    ("model", DummyRegressor(strategy="mean"))])

dummy_model.fit(X_train, y_train)

dummy_pred = dummy_model.predict(X_test)

print(evaluation_model("Dummy Mean", y_test, dummy_pred))

Dummy Mean: home MAE: 1.2591058092948026, away MAE: 0.9810486202160958, home RMSE: 1.6379327490749456, away RMSE: 1.3163108738023854, home R2: -0.011529825866999355, away R2: -0.003961418036618314, home poisson deviance: 1.6508754687728229, away poisson deviance: 1.5050459545307169, exact score accuracy: 0.073991711311028, result accuracy: 0.47670069746285254, precision: 0.2272435549615701, recall: 0.47670069746285254, Fscore: 0.3077719883954839


# 3. Ridge Regression

In [245]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))])

ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)

print(evaluation_model("Ridge Regression", y_test, ridge_pred))

Ridge Regression: home MAE: 1.046468673522382, away MAE: 0.8491968742762367, home RMSE: 1.410385434488145, away RMSE: 1.1703736489879, home R2: 0.2499984022337507, away R2: 0.20631320353671778, home poisson deviance: 1.4341222979459105, away poisson deviance: 1.384928581870525, exact score accuracy: 0.11765895077327403, result accuracy: 0.5391691094713433, precision: 0.5850092734292137, recall: 0.5391691094713433, Fscore: 0.5430345350956202


# 4. KNN Regressio

In [246]:
from sklearn.neighbors import KNeighborsRegressor

knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsRegressor(
        n_neighbors=50,
        weights="distance"))])

knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)

print(evaluation_model("KNN Regression", y_test, knn_pred))

KNN Regression: home MAE: 1.0452307583060099, away MAE: 0.8509490715083843, home RMSE: 1.3961769814302243, away RMSE: 1.1510016393565055, home R2: 0.2650335625076048, away R2: 0.23236994588724968, home poisson deviance: 1.2432396324378947, away poisson deviance: 1.1935060402429651, exact score accuracy: 0.10846052764581017, result accuracy: 0.5316890730819771, precision: 0.5972893345038526, recall: 0.5316890730819771, Fscore: 0.5393580353786157


# 5. Random Forest Regressor

In [247]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1))])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(evaluation_model("Random Forest", y_test, rf_pred))

Random Forest: home MAE: 1.0407651878161763, away MAE: 0.8523226526819918, home RMSE: 1.3827536236084006, away RMSE: 1.1402888716177282, home R2: 0.2790980991588593, away R2: 0.24659264200997255, home poisson deviance: 1.218486497434281, away poisson deviance: 1.1753631569456136, exact score accuracy: 0.10613565147073689, result accuracy: 0.5487718588901244, precision: 0.5951358293121797, recall: 0.5487718588901244, Fscore: 0.5571285725576983


# 6. extra trees regressor

In [248]:
from sklearn.ensemble import ExtraTreesRegressor

et_model = Pipeline([
    ("model", ExtraTreesRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1))])

et_model.fit(X_train, y_train)

et_pred = et_model.predict(X_test)

print(evaluation_model("Extra Trees", y_test, et_pred))

Extra Trees: home MAE: 1.041109751044225, away MAE: 0.8469439646807524, home RMSE: 1.3760877198285104, away RMSE: 1.1377516983859912, home R2: 0.2860319153741603, away R2: 0.24994161511970292, home poisson deviance: 1.2168995410477006, away poisson deviance: 1.1717556965098317, exact score accuracy: 0.1053269988881027, result accuracy: 0.5510967350651976, precision: 0.5992737446819222, recall: 0.5510967350651976, Fscore: 0.5555306055318341


# 6.1. tuning extra trees

In [249]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

et_pipeline = Pipeline([
    ("model", ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1))])

param_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [8, 12, 16, None],
    "model__min_samples_leaf": [5, 10, 20],
    "model__min_samples_split": [2, 10, 20],}

tscv = TimeSeriesSplit(n_splits=5)

et_search = GridSearchCV(
    estimator=et_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    n_jobs=-1,
    verbose=1)

et_search.fit(X_train, y_train)

print("Best params:", et_search.best_params_)
print("Best CV score:", et_search.best_score_)

et_best_pred = et_search.best_estimator_.predict(X_test)

print(evaluation_model("Tuned Extra Trees", y_test, et_best_pred))

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best params: {'model__max_depth': 16, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2, 'model__n_estimators': 500}
Best CV score: -0.9971221222547637
Tuned Extra Trees: home MAE: 1.0373168318412964, away MAE: 0.8481077470112812, home RMSE: 1.374336825483005, away RMSE: 1.1388156042933002, home R2: 0.2878476243288397, away R2: 0.24853820759839618, home poisson deviance: 1.2134386297986586, away poisson deviance: 1.1720374471085135, exact score accuracy: 0.10755079349034671, result accuracy: 0.5451329222682705, precision: 0.5984115281939372, recall: 0.5451329222682705, Fscore: 0.5536046665559747


# 7. HistGradientBoosting with Poisson Loss

In [250]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb_poisson_model = Pipeline([
    ("model", MultiOutputRegressor(
        HistGradientBoostingRegressor(
            loss="poisson",
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=42)))])

hgb_poisson_model.fit(X_train, y_train)

hgb_poisson_pred = hgb_poisson_model.predict(X_test)

print(evaluation_model("HistGradientBoosting Poisson", y_test, hgb_poisson_pred))

HistGradientBoosting Poisson: home MAE: 1.0349010756695467, away MAE: 0.8477992526834406, home RMSE: 1.387993030729229, away RMSE: 1.1370246373081974, home R2: 0.27362459410181283, away R2: 0.2508999332024169, home poisson deviance: 1.2230207304794047, away poisson deviance: 1.1684927553251536, exact score accuracy: 0.11129081168502981, result accuracy: 0.5452340038410998, precision: 0.6011298454680338, recall: 0.5452340038410998, Fscore: 0.5576308163796926


| رتبه | Model             | Result Accuracy ↑ |    Fscore ↑ | Exact Score Acc ↑ |   Avg MAE ↓ |  Avg RMSE ↓ |     Avg R2 ↑ | Avg Poisson Dev ↓ |
| ---: | ----------------- | ----------------: |  ---------: | ----------------: |  ---------: |  ---------: | -----------: | ----------------: |
|    1 | Extra Trees       |            0.5511 |      0.5555 |            0.1053 |      0.9440 |      1.2569 |       0.2680 |            1.1943 |
|    2 | Random Forest     |            0.5488 |      0.5571 |            0.1061 |      0.9465 |      1.2615 |       0.2628 |            1.1969 |
|    3 | HGB Poisson       |            0.5452 |    `0.5576` |            0.1113 |    `0.9414` |      1.2625 |       0.2623 |            1.1958 |
|    4 | Tuned Extra Trees |            0.5451 |      0.5536 |            0.1076 |      0.9427 |    `1.2566` |     `0.2682` |          `1.1927` |
|    5 | Ridge             |            0.5392 |      0.5430 |          `0.1177` |      0.9478 |      1.2904 |       0.2282 |            1.4095 |
|    6 | KNN               |            0.5317 |      0.5394 |            0.1085 |      0.9481 |      1.2736 |       0.2487 |            1.2184 |
|    7 | Poisson           |            0.5220 |      0.5238 |            0.1052 |      0.9554 |      1.2918 |       0.2260 |            1.2429 |
|    8 | Dummy Mean        |            0.4767 |      0.3078 |            0.0740 |      1.1201 |      1.4771 |      -0.0077 |            1.5780 |


# 2026 world cup

In [251]:
wc_predict_df = schedule_2026.copy()

wc_predict_df = wc_predict_df.rename(columns={
    "Date": "date"
})

wc_predict_df["date"] = pd.to_datetime(wc_predict_df["date"])
wc_predict_df["tournament"] = "FIFA World Cup"
wc_predict_df["neutral"] = 1
wc_predict_df["year"] = wc_predict_df["date"].dt.year
wc_predict_df["country"] = "neutral"

wc_predict_df["is_home_country"] = 0

wc_predict_df.head()

,Round,Day,date,Time,Score,Referee,Notes,Year,home_team,away_team,tournament,neutral,year,country,is_home_country
0,Group stage,Thu,2026-06-11,13:00 (22:00),NaN,NaN,NaN,2026,Mexico,South Africa,FIFA World Cup,1,2026,neutral,0
1,Group stage,Thu,2026-06-11,20:00 (05:00),NaN,NaN,NaN,2026,Korea Republic,Czechia,FIFA World Cup,1,2026,neutral,0
2,Group stage,Fri,2026-06-12,15:00 (22:00),NaN,NaN,NaN,2026,Canada,Bosnia-Herzegovina,FIFA World Cup,1,2026,neutral,0
3,Group stage,Fri,2026-06-12,18:00 (04:00),NaN,NaN,NaN,2026,United States,Paraguay,FIFA World Cup,1,2026,neutral,0
4,Group stage,Sat,2026-06-13,12:00 (22:00),NaN,NaN,NaN,2026,Qatar,Switzerland,FIFA World Cup,1,2026,neutral,0


In [252]:
def clean_team_name(s):
    return str(s).replace("\xa0", " ").strip()

team_name_map = {
    "Republic of Ireland": "Ireland",
    "DR Congo": "Democratic Republic of Congo",
    "Czech Republic": "Czechia",
    "German DR": "East Germany",
    "North Macedonia": "Macedonia",
    "Eswatini": "Swaziland",
    "Réunion": "Reunion",
    "Vietnam Republic": "South Vietnam",}

def map_team_name(team):
    team = clean_team_name(team)
    return team_name_map.get(team, team)

wc_predict_df["home_team_key"] = wc_predict_df["home_team"].apply(map_team_name)
wc_predict_df["away_team_key"] = wc_predict_df["away_team"].apply(map_team_name)

model_df["home_team_key"] = model_df["home_team"].apply(map_team_name)
model_df["away_team_key"] = model_df["away_team"].apply(map_team_name)

In [253]:
elo["team"] = elo["team"].apply(clean_team_name)
elo["date"] = pd.to_datetime(elo["date"], format="mixed")
elo["rating"] = pd.to_numeric(elo["rating"])

def latest_elo_before(team, match_date):
    team_elo = elo[(elo["team"].eq(team)) & (elo["date"] < match_date)].sort_values("date")
    
    if team_elo.empty:
        return 2000
    
    return team_elo["rating"].iloc[-1]

wc_predict_df["home_elo_before"] = wc_predict_df.apply(lambda row: latest_elo_before(row["home_team_key"], row["date"]), axis=1)

wc_predict_df["away_elo_before"] = wc_predict_df.apply(lambda row: latest_elo_before(row["away_team_key"], row["date"]), axis=1)

wc_predict_df["elo_diff"] = wc_predict_df["home_elo_before"] - wc_predict_df["away_elo_before"]

wc_predict_df["home_elo_missing"] = 0
wc_predict_df["away_elo_missing"] = 0

In [255]:
history_home = model_df[["date", "home_team_key", "home_goals", "away_goals"]].rename(columns={
    "home_team_key": "team",
    "home_goals": "goals_for",
    "away_goals": "goals_against"})

history_away = model_df[["date", "away_team_key", "away_goals", "home_goals"]].rename(columns={
    "away_team_key": "team",
    "away_goals": "goals_for",
    "home_goals": "goals_against"})

team_history = pd.concat([history_home, history_away], ignore_index=True)
team_history = team_history.sort_values(["team", "date"])

team_history["points"] = np.where(
    team_history["goals_for"] > team_history["goals_against"], 3,
    np.where(team_history["goals_for"] == team_history["goals_against"], 1, 0))

AVG_GOALS = model_df[["home_goals", "away_goals"]].stack().mean()
AVG_POINTS = team_history["points"].mean()
AVG_REST = 30

In [256]:
def team_recent_features(team, match_date):
    past = team_history[(team_history["team"].eq(team)) & (team_history["date"] < match_date)].sort_values("date")
    
    last_5 = past.tail(5)
    
    if last_5.empty:
        avg_for = AVG_GOALS
        avg_against = AVG_GOALS
        avg_points = AVG_POINTS
        rest_days = AVG_REST
    else:
        avg_for = last_5["goals_for"].mean()
        avg_against = last_5["goals_against"].mean()
        avg_points = last_5["points"].mean()
        rest_days = (match_date - past["date"].iloc[-1]).days
    
    return avg_for, avg_against, avg_points, rest_days

In [257]:
home_features = wc_predict_df.apply(lambda row: team_recent_features(row["home_team_key"], row["date"]), axis=1)

away_features = wc_predict_df.apply(lambda row: team_recent_features(row["away_team_key"], row["date"]), axis=1)

wc_predict_df[[
        "home_avg_goals_for_last_5",
        "home_avg_goals_against_last_5",
        "home_avg_points_last_5",
        "home_days_since_last_match",]
] = pd.DataFrame(home_features.tolist(), index=wc_predict_df.index)

wc_predict_df[[
        "away_avg_goals_for_last_5",
        "away_avg_goals_against_last_5",
        "away_avg_points_last_5",
        "away_days_since_last_match",]
] = pd.DataFrame(away_features.tolist(), index=wc_predict_df.index)

# predict

In [ ]:
matches_2026 = pd.read_csv("data/FIFA World Cup 2026 - Live Results & Updated Stats/matches.csv")
matches_2026.head()

In [ ]:
teams_2026 = pd.read_csv("data/FIFA World Cup 2026 - Live Results & Updated Stats/teams.csv")
teams_2026.head()

In [286]:
matches_2026["date"] = pd.to_datetime(matches_2026["date"])

# -----------------------------
# 2) Add home/away team names from teams.csv
# -----------------------------
wc_model_ready_df = (
    matches_2026
    .merge(
        teams_2026[["team_id", "team_name"]],
        left_on="home_team_id",
        right_on="team_id",
        how="left"
    )
    .rename(columns={"team_name": "home_team"})
    .drop(columns=["team_id"])
    .merge(
        teams_2026[["team_id", "team_name"]],
        left_on="away_team_id",
        right_on="team_id",
        how="left"
    )
    .rename(columns={"team_name": "away_team"})
    .drop(columns=["team_id"])
)

wc_model_ready_df = wc_model_ready_df[
    ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "status"]
].copy()

# scores: completed games have numbers, future games stay NaN
wc_model_ready_df["home_score"] = pd.to_numeric(wc_model_ready_df["home_score"], errors="coerce")
wc_model_ready_df["away_score"] = pd.to_numeric(wc_model_ready_df["away_score"], errors="coerce")

1. features

In [287]:
# -----------------------------
# 3) Team name cleaning/mapping
# -----------------------------
def clean_team_name(s):
    return str(s).replace("\xa0", " ").strip()

team_name_map = {
    "DR Congo": "Democratic Republic of Congo",
    "Congo DR": "Democratic Republic of Congo",
    "Czech Republic": "Czechia",
    "Republic of Ireland": "Ireland",
    "German DR": "East Germany",
    "North Macedonia": "Macedonia",
    "Eswatini": "Swaziland",
    "Réunion": "Reunion",
    "Vietnam Republic": "South Vietnam",
}

def map_team_name(team):
    team = clean_team_name(team)
    return team_name_map.get(team, team)

wc_model_ready_df["home_team_key"] = wc_model_ready_df["home_team"].apply(map_team_name)
wc_model_ready_df["away_team_key"] = wc_model_ready_df["away_team"].apply(map_team_name)

model_df["date"] = pd.to_datetime(model_df["date"])
model_df["home_team_key"] = model_df["home_team"].apply(map_team_name)
model_df["away_team_key"] = model_df["away_team"].apply(map_team_name)

# -----------------------------
# 4) Context features
# -----------------------------
wc_model_ready_df["neutral"] = 1
wc_model_ready_df["is_home_country"] = 0
wc_model_ready_df["year"] = wc_model_ready_df["date"].dt.year

2. elo

In [288]:
# -----------------------------
# 5) Elo features
# -----------------------------
elo = elo.copy()
elo["date"] = pd.to_datetime(elo["date"], format="mixed")
elo["team"] = elo["team"].apply(clean_team_name)
elo["team_key"] = elo["team"].apply(map_team_name)
elo["rating"] = pd.to_numeric(elo["rating"], errors="coerce")

def latest_elo_before(team, match_date):
    past = elo[
        (elo["team_key"].eq(team)) &
        (elo["date"] < match_date)
    ].sort_values("date")
    
    if past.empty:
        return 2000, 1
    
    return past["rating"].iloc[-1], 0

home_elo_info = wc_model_ready_df.apply(
    lambda row: latest_elo_before(row["home_team_key"], row["date"]),
    axis=1
)

away_elo_info = wc_model_ready_df.apply(
    lambda row: latest_elo_before(row["away_team_key"], row["date"]),
    axis=1
)

wc_model_ready_df["home_elo_before"] = [x[0] for x in home_elo_info]
wc_model_ready_df["home_elo_missing"] = [x[1] for x in home_elo_info]

wc_model_ready_df["away_elo_before"] = [x[0] for x in away_elo_info]
wc_model_ready_df["away_elo_missing"] = [x[1] for x in away_elo_info]

wc_model_ready_df["elo_diff"] = (
    wc_model_ready_df["home_elo_before"] -
    wc_model_ready_df["away_elo_before"]
)

3. recent features

In [293]:
# -----------------------------
# 6) Build historical team-level data
# -----------------------------
wc_start_date = wc_model_ready_df["date"].min()

history_df = model_df[model_df["date"] < wc_start_date].copy()

history_home = history_df[["date", "home_team_key", "home_goals", "away_goals"]].rename(columns={
    "home_team_key": "team",
    "home_goals": "goals_for",
    "away_goals": "goals_against",
})

history_away = history_df[["date", "away_team_key", "away_goals", "home_goals"]].rename(columns={
    "away_team_key": "team",
    "away_goals": "goals_for",
    "home_goals": "goals_against",
})

team_history = pd.concat([history_home, history_away], ignore_index=True)

# add completed 2026 World Cup games too
completed_wc = wc_model_ready_df.dropna(subset=["home_score", "away_score"]).copy()

wc_home_history = completed_wc[["date", "home_team_key", "home_score", "away_score"]].rename(columns={
    "home_team_key": "team",
    "home_score": "goals_for",
    "away_score": "goals_against",
})

wc_away_history = completed_wc[["date", "away_team_key", "away_score", "home_score"]].rename(columns={
    "away_team_key": "team",
    "away_score": "goals_for",
    "home_score": "goals_against",
})

team_history = pd.concat([team_history, wc_home_history, wc_away_history], ignore_index=True)

team_history["points"] = np.where(
    team_history["goals_for"] > team_history["goals_against"], 3,
    np.where(team_history["goals_for"] == team_history["goals_against"], 1, 0)
)

team_history = team_history.sort_values(["team", "date"])

AVG_GOALS = model_df[["home_goals", "away_goals"]].stack().mean()
AVG_POINTS = team_history["points"].mean()
AVG_REST = 30

def recent_features(team, match_date):
    past = team_history[
        (team_history["team"].eq(team)) &
        (team_history["date"] < match_date)
    ].sort_values("date")
    
    last_5 = past.tail(5)
    
    if last_5.empty:
        return AVG_GOALS, AVG_GOALS, AVG_POINTS, AVG_REST
    
    avg_goals_for = last_5["goals_for"].mean()
    avg_goals_against = last_5["goals_against"].mean()
    avg_points = last_5["points"].mean()
    days_since_last_match = (match_date - past["date"].iloc[-1]).days
    
    return avg_goals_for, avg_goals_against, avg_points, days_since_last_match

In [294]:
# -----------------------------
# 7) Add recent features to World Cup dataframe
# -----------------------------
home_recent = wc_model_ready_df.apply(
    lambda row: recent_features(row["home_team_key"], row["date"]),
    axis=1
)

away_recent = wc_model_ready_df.apply(
    lambda row: recent_features(row["away_team_key"], row["date"]),
    axis=1
)

wc_model_ready_df[
    [
        "home_avg_goals_for_last_5",
        "home_avg_goals_against_last_5",
        "home_avg_points_last_5",
        "home_days_since_last_match",
    ]
] = pd.DataFrame(home_recent.tolist(), index=wc_model_ready_df.index)

wc_model_ready_df[
    [
        "away_avg_goals_for_last_5",
        "away_avg_goals_against_last_5",
        "away_avg_points_last_5",
        "away_days_since_last_match",
    ]
] = pd.DataFrame(away_recent.tolist(), index=wc_model_ready_df.index)

In [295]:
# -----------------------------
# 8) Final check
# -----------------------------
print("Missing in feature_cols:")
print(wc_model_ready_df[feature_cols].isna().sum())

wc_model_ready_df[
    ["date", "home_team", "away_team", "home_score", "away_score"] + feature_cols
].head(20)

Missing in feature_cols:
home_elo_before                  0
away_elo_before                  0
elo_diff                         0
home_elo_missing                 0
away_elo_missing                 0
home_avg_goals_for_last_5        0
away_avg_goals_for_last_5        0
home_avg_goals_against_last_5    0
away_avg_goals_against_last_5    0
home_avg_points_last_5           0
away_avg_points_last_5           0
home_days_since_last_match       0
away_days_since_last_match       0
neutral                          0
is_home_country                  0
year                             0
dtype: int64


,date,home_team,away_team,home_score,away_score,home_elo_before,away_elo_before,elo_diff,home_elo_missing,away_elo_missing,...,away_avg_goals_for_last_5,home_avg_goals_against_last_5,away_avg_goals_against_last_5,home_avg_points_last_5,away_avg_points_last_5,home_days_since_last_match,away_days_since_last_match,neutral,is_home_country,year
0,2026-06-11,Mexico,South Africa,2.0,0.0,1835.0,1531.0,304.0,0,0,...,0.800000,0.400000,1.000000,2.200000,1.000000,7,5,1,0,2026
1,2026-06-11,South Korea,Czechia,2.0,1.0,1784.0,1731.0,53.0,0,0,...,3.400000,1.000000,1.200000,1.800000,2.200000,8,7,1,0,2026
2,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,1802.0,1571.0,231.0,0,0,...,0.800000,0.600000,0.800000,1.800000,1.000000,7,6,1,0,2026
3,2026-06-12,USA,Paraguay,4.0,1.0,2000.0,1833.0,167.0,1,0,...,1.800000,1.469595,1.000000,1.386295,1.800000,30,7,1,0,2026
4,2026-06-13,Qatar,Switzerland,1.0,1.0,1427.0,1897.0,-470.0,0,0,...,1.800000,1.200000,1.400000,0.400000,1.200000,7,7,1,0,2026
5,2026-06-13,Brazil,Morocco,1.0,1.0,1979.0,1830.0,149.0,0,0,...,2.600000,1.400000,0.600000,2.000000,2.200000,7,6,1,0,2026
6,2026-06-13,Haiti,Scotland,0.0,1.0,1542.0,1790.0,-248.0,0,0,...,2.400000,0.800000,1.000000,1.400000,1.800000,8,7,1,0,2026
7,2026-06-13,Australia,Türkiye,2.0,0.0,1774.0,2000.0,-226.0,0,1,...,1.469595,1.200000,1.469595,1.400000,1.386295,7,30,1,0,2026
8,2026-06-14,Germany,Curaçao,7.0,1.0,1910.0,1467.0,443.0,0,0,...,1.200000,1.000000,2.200000,3.000000,0.800000,8,8,1,0,2026
9,2026-06-14,Netherlands,Japan,2.0,2.0,1959.0,1878.0,81.0,0,0,...,1.600000,0.800000,0.000000,2.000000,3.000000,6,14,1,0,2026


In [296]:
X_wc = wc_model_ready_df[feature_cols]

models_for_prediction = {
    "ridge": ridge_model,
    "random_forest": rf_model,
    "extra_trees": et_model,
    "hgb_poisson": hgb_poisson_model,
}

for model_name, model in models_for_prediction.items():
    pred_goals = model.predict(X_wc)
    pred_goals = np.clip(pred_goals, 0, None)

    pred_home_score = np.rint(pred_goals[:, 0]).astype(int)
    pred_away_score = np.rint(pred_goals[:, 1]).astype(int)

    wc_model_ready_df[f"pred_score_{model_name}"] = (
        pred_home_score.astype(str) + "-" + pred_away_score.astype(str)
    )

* home-away

In [ ]:
display_cols = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
]

prediction_score_cols = [
    col for col in wc_model_ready_df.columns
    if col.startswith("pred_score_")
]

wc_model_ready_df[display_cols + prediction_score_cols].head()

,date,home_team,away_team,home_score,away_score,pred_score_ridge,pred_score_random_forest,pred_score_extra_trees,pred_score_hgb_poisson
0,2026-06-11,Mexico,South Africa,2.0,0.0,2-1,2-1,2-1,2-1
1,2026-06-11,South Korea,Czechia,2.0,1.0,1-1,1-1,1-1,1-1
2,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,1-1,2-1,2-1,1-1
3,2026-06-12,USA,Paraguay,4.0,1.0,0-2,1-2,1-2,1-3
4,2026-06-13,Qatar,Switzerland,1.0,1.0,0-2,1-2,1-2,1-2
...,...,...,...,...,...,...,...,...,...
67,2026-06-28,Algeria,Austria,NaN,NaN,1-1,1-2,1-1,1-1
68,2026-06-29,Colombia,Portugal,NaN,NaN,1-1,2-1,1-1,1-1
69,2026-06-29,Congo DR,Uzbekistan,NaN,NaN,1-1,1-1,1-1,1-1
70,2026-06-29,Panama,England,NaN,NaN,0-2,1-2,1-2,1-2


# saved to do changes

In [ ]:
Path("outputs").mkdir(exist_ok=True)
wc_model_ready_df.to_csv("outputs/world_cup_2026_predictions.csv", index=False)

In [ ]:
wc_predictions_edited = pd.read_csv(
    "outputs/world_cup_2026_predictions.csv",
    encoding="utf-8-sig")

In [308]:
wc_predictions_edited[display_cols + prediction_score_cols].head(42)

,date,home_team,away_team,home_score,away_score,pred_score_ridge,pred_score_random_forest,pred_score_extra_trees,pred_score_hgb_poisson
0,2026-06-11,Mexico,South Africa,2.0,0.0,2-1,2-1,2-1,2-1
1,2026-06-11,South Korea,Czechia,2.0,1.0,1-1,1-1,1-1,1-1
2,2026-06-12,Canada,Bosnia and Herzegovina,1.0,1.0,1-1,2-1,2-1,1-1
3,2026-06-12,USA,Paraguay,4.0,1.0,0-2,1-2,1-2,1-3
4,2026-06-13,Qatar,Switzerland,1.0,1.0,0-2,1-2,1-2,1-2
5,2026-06-13,Brazil,Morocco,1.0,1.0,1-1,2-1,2-1,1-1
6,2026-06-13,Haiti,Scotland,0.0,1.0,1-1,1-2,1-1,1-2
7,2026-06-13,Australia,Türkiye,2.0,0.0,2-1,4-1,3-1,4-1
8,2026-06-14,Germany,Curaçao,7.0,1.0,3-1,3-1,3-1,3-1
9,2026-06-14,Netherlands,Japan,2.0,2.0,1-1,1-1,1-1,1-1


In [309]:
wc_predictions_edited[display_cols + prediction_score_cols].tail(30)

,date,home_team,away_team,home_score,away_score,pred_score_ridge,pred_score_random_forest,pred_score_extra_trees,pred_score_hgb_poisson
42,2026-06-22,Argentina,Austria,2.0,0.0,2-1,2-1,2-1,2-1
43,2026-06-22,Jordan,Algeria,1.0,2.0,1-2,1-2,1-1,1-2
44,2026-06-23,Portugal,Uzbekistan,5.0,0.0,2-1,2-1,2-1,2-1
45,2026-06-23,Colombia,Congo DR,1.0,0.0,2-1,2-1,2-1,2-1
46,2026-06-23,England,Ghana,0.0,0.0,3-0,3-0,2-1,3-0
47,2026-06-23,Panama,Croatia,0.0,1.0,1-2,1-2,1-2,1-2
48,2026-06-24,Czechia,Mexico,0.0,3.0,1-2,1-1,1-1,1-2
49,2026-06-24,South Africa,South Korea,1.0,0.0,0-1,1-2,1-2,1-2
50,2026-06-24,Switzerland,Canada,2.0,1.0,1-1,1-1,1-1,1-1
51,2026-06-24,Bosnia and Herzegovina,Qatar,3.0,1.0,2-1,2-1,2-1,2-1
